# Antall bøker hvor over 20% av boka er et samisk språk

In [1]:
import pandas as pd

lang_code_map = {
    "sme": "nordsamisk",
    "smj": "lulesamisk",
    "sma": "sørsamisk",
    "smn": "enaresamisk",
    "sms": "skoltesamisk",
    "smi": "samiske språk",
}

threshold = 0.2

for detection_model in ["gielladetect", "glotlid"]:
    df = pd.read_json(
        f"../data/sámi_language_statistics/{detection_model}/tokens_stats.jsonl",
        lines=True,
        orient="records",
    )
    print(f"------------------------{detection_model}------------------------")
    for lang_code, lang_name in lang_code_map.items():

        def lang_in_lang_list(lang_list: list[str] | None) -> bool:
            return lang_list is not None and lang_code in lang_list

        detected_lang_code = df[f"detected_{lang_code}_share"] > threshold
        categorized_by_library = df.library_languages.apply(lang_in_lang_list)

        print(
            f"Ifølge {detection_model} er {detected_lang_code.sum()} bøker på {lang_name} (over {threshold * 100}% av avsnittene er på språket)"
        )
        print(
            f"Ifølge bibliotekets metadata er {categorized_by_library.sum()} bøker på {lang_name}"
        )
        print(
            f"\n{(detected_lang_code & categorized_by_library).sum()} bøker overlapper (kategorisert {lang_code} av både modell og bibliotek)"
        )

        print(
            f"{(detected_lang_code & ~categorized_by_library).sum()} bøker er kun kategorisert {lang_code} av {detection_model}"
        )
        print(
            f"{(~detected_lang_code & categorized_by_library).sum()} bøker er kun kategorisert {lang_code} av biblioteket \n\n"
        )

    print(" ")

------------------------gielladetect------------------------
Ifølge gielladetect er 2479 bøker på nordsamisk (over 20.0% av avsnittene er på språket)
Ifølge bibliotekets metadata er 2585 bøker på nordsamisk

2318 bøker overlapper (kategorisert sme av både modell og bibliotek)
161 bøker er kun kategorisert sme av gielladetect
267 bøker er kun kategorisert sme av biblioteket 


Ifølge gielladetect er 478 bøker på lulesamisk (over 20.0% av avsnittene er på språket)
Ifølge bibliotekets metadata er 399 bøker på lulesamisk

350 bøker overlapper (kategorisert smj av både modell og bibliotek)
128 bøker er kun kategorisert smj av gielladetect
49 bøker er kun kategorisert smj av biblioteket 


Ifølge gielladetect er 429 bøker på sørsamisk (over 20.0% av avsnittene er på språket)
Ifølge bibliotekets metadata er 448 bøker på sørsamisk

395 bøker overlapper (kategorisert sma av både modell og bibliotek)
34 bøker er kun kategorisert sma av gielladetect
53 bøker er kun kategorisert sma av biblioteket

## Inspiser diffen på lulesamisk 

In [2]:
from IPython.display import HTML

df = pd.read_json(
    "../data/sámi_language_statistics/gielladetect/tokens_stats.jsonl",
    lines=True,
    orient="records",
)

df["url"] = df.urn.apply(lambda urn: f"https://www.nb.no/items/{urn}")


def smj_in_langlist(lang_list: list[str] | None) -> bool:
    return lang_list is not None and "smj" in lang_list


categorized_smj_by_library = df[df.library_languages.apply(smj_in_langlist)]
categorized_smj_by_gielladetect = df[df.detected_smj_share > 0.2]

difference = categorized_smj_by_gielladetect[
    ~categorized_smj_by_gielladetect.index.isin(categorized_smj_by_library.index)
]


difference = difference.sort_values(by="detected_smj_share", ascending=False)[
    [
        "title",
        "year",
        "detected_smj_share",
        "num_paragraphs",
        "library_languages",
        "url",
        "all_detected_languages",
    ]
]

# For at url-ene skal være klikkbare fra notebooken
HTML(difference.to_html(render_links=True, escape=False))


,title,year,detected_smj_share,num_paragraphs,library_languages,url,all_detected_languages
1769,Gotthard Erichsens beste,1900.0,1.0000,1,[zxx],https://www.nb.no/items/URN:NBN:no-nb_digibok_2012060108117,{'smj': 1.0}
4266,Tre motetter : for enstemt kor og orgel,1977.0,1.0000,1,[nob],https://www.nb.no/items/URN:NBN:no-nb_digibok_2015070658062,{'smj': 1.0}
5655,Dinosaurer,2016.0,1.0000,1,[nob],https://www.nb.no/items/URN:NBN:no-nb_digibok_2021082548625,{'smj': 1.0}
475,Gutten og fuglen,1984.0,1.0000,1,[nob],https://www.nb.no/items/URN:NBN:no-nb_digibok_2013071205043,{'smj': 1.0}
5278,"O, Oarjjevuodna",1983.0,0.9597,19,[sme],https://www.nb.no/items/URN:NBN:no-nb_digibok_2018042448549,"{'nno': 0.0403, 'smj': 0.9597}"
726,Biejve näjjda,2020.0,0.9540,32,"[smi, nob]",https://www.nb.no/items/URN:NBN:no-nb_digibok_2020040648085,"{'est': 0.0143, 'nno': 0.0155, 'nob': 0.01, 'smj': 0.9540000000000001, 'sms': 0.006200000000000001}"
1145,Pluppa ja sarrva,2022.0,0.9467,42,"[sje, swe]",https://www.nb.no/items/URN:NBN:no-nb_pliktmonografi_000017150,"{'nno': 0.009000000000000001, 'nob': 0.013600000000000001, 'sma': 0.012700000000000001, 'smj': 0.9467000000000001, 'swe': 0.0181}"
5203,SIM : máná giellaáddejupmi : sámegiela impressiiva morfologalaš teasta : SIM : sámegiela impressijvva morfologalalasj gæhttjalibme,2014.0,0.9422,8,[sma],https://www.nb.no/items/URN:NBN:no-nb_digibok_2017112048532,"{'kal': 0.057800000000000004, 'smj': 0.9422}"
4125,Suohtas Pluppa ludne,2025.0,0.9388,45,"[sje, swe]",https://www.nb.no/items/URN:NBN:no-nb_pliktmonografi_000044827,"{'est': 0.008700000000000001, 'hbs': 0.0079, 'nno': 0.0143, 'nob': 0.0119, 'smj': 0.9388000000000001, 'sms': 0.0079, 'swe': 0.0103}"
2172,Vihkele baakoeh maanide - : Darjoeh dov bööremes!,2021.0,0.9384,26,[sma],https://www.nb.no/items/URN:NBN:no-nb_digibok_2024121348023,"{'eng': 0.0616, 'smj': 0.9384}"
